**Materia:** Tecnologías Emergentes — Primavera 2026  
**Profesor:** Mtro. Rafael Pérez Aguirre

# 🗄️ SQL Agent

Un **SQL Agent** es un agente capaz de interactuar con bases de datos SQL de forma autónoma. En lugar de escribir consultas manualmente, el agente recibe una pregunta en lenguaje natural, genera la consulta SQL apropiada, la ejecuta contra la base de datos y devuelve una respuesta comprensible.

Esto es extremadamente útil cuando queremos que usuarios no técnicos puedan explorar datos sin conocer SQL, o cuando queremos construir sistemas que razonen sobre datos estructurados.

📖 [Documentación oficial: SQL Agent](https://docs.langchain.com/oss/python/langchain/sql-agent)

## Configuración

Cargar y/o verificar las variables de entorno necesarias

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

variables_requeridas = ["GEMINI_API_KEY"]
for var in variables_requeridas:
    valor = os.environ.get(var)
    if valor:
        print(f"✅ {var} = {valor[:8]}...{valor[-4:]}")
    else:
        print(f"❌ {var} no está definida")

## Preparar la base de datos

Vamos a crear una base de datos SQLite de ejemplo con información de una **tienda de videojuegos**. Tendremos tres tablas relacionadas: `categorias`, `videojuegos` y `ventas`.

In [ ]:
import sqlite3

conexion = sqlite3.connect("tienda_videojuegos.db")
cursor = conexion.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS ventas;
DROP TABLE IF EXISTS videojuegos;
DROP TABLE IF EXISTS categorias;

CREATE TABLE categorias (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL
);

CREATE TABLE videojuegos (
    id INTEGER PRIMARY KEY,
    titulo TEXT NOT NULL,
    categoria_id INTEGER,
    precio REAL,
    anio_lanzamiento INTEGER,
    calificacion REAL,
    FOREIGN KEY (categoria_id) REFERENCES categorias(id)
);

CREATE TABLE ventas (
    id INTEGER PRIMARY KEY,
    videojuego_id INTEGER,
    cantidad INTEGER,
    fecha TEXT,
    FOREIGN KEY (videojuego_id) REFERENCES videojuegos(id)
);

-- Categorías
INSERT INTO categorias VALUES (1, 'Acción');
INSERT INTO categorias VALUES (2, 'RPG');
INSERT INTO categorias VALUES (3, 'Aventura');
INSERT INTO categorias VALUES (4, 'Deportes');
INSERT INTO categorias VALUES (5, 'Estrategia');

-- Videojuegos
INSERT INTO videojuegos VALUES (1, 'Zelda: Tears of the Kingdom', 3, 59.99, 2023, 9.5);
INSERT INTO videojuegos VALUES (2, 'Elden Ring', 2, 49.99, 2022, 9.8);
INSERT INTO videojuegos VALUES (3, 'God of War Ragnarök', 1, 59.99, 2022, 9.4);
INSERT INTO videojuegos VALUES (4, 'FIFA 25', 4, 69.99, 2024, 7.2);
INSERT INTO videojuegos VALUES (5, 'Civilization VII', 5, 49.99, 2025, 8.8);
INSERT INTO videojuegos VALUES (6, 'Baldurs Gate 3', 2, 59.99, 2023, 9.7);
INSERT INTO videojuegos VALUES (7, 'Spider-Man 2', 1, 69.99, 2023, 9.0);
INSERT INTO videojuegos VALUES (8, 'Hollow Knight: Silksong', 3, 39.99, 2025, 9.3);
INSERT INTO videojuegos VALUES (9, 'Mario Kart 9', 4, 59.99, 2025, 8.5);
INSERT INTO videojuegos VALUES (10, 'Final Fantasy VII Rebirth', 2, 69.99, 2024, 9.1);

-- Ventas
INSERT INTO ventas VALUES (1, 1, 150, '2025-01-15');
INSERT INTO ventas VALUES (2, 2, 200, '2025-01-20');
INSERT INTO ventas VALUES (3, 3, 180, '2025-02-10');
INSERT INTO ventas VALUES (4, 4, 300, '2025-02-14');
INSERT INTO ventas VALUES (5, 5, 90, '2025-03-01');
INSERT INTO ventas VALUES (6, 6, 250, '2025-01-25');
INSERT INTO ventas VALUES (7, 7, 170, '2025-02-28');
INSERT INTO ventas VALUES (8, 8, 120, '2025-03-05');
INSERT INTO ventas VALUES (9, 9, 210, '2025-03-10');
INSERT INTO ventas VALUES (10, 10, 160, '2025-01-30');
INSERT INTO ventas VALUES (11, 1, 80, '2025-02-20');
INSERT INTO ventas VALUES (12, 2, 95, '2025-03-08');
INSERT INTO ventas VALUES (13, 6, 110, '2025-03-12');
""")

conexion.commit()
conexion.close()

print("- Base de datos 'tienda_videojuegos.db' creada exitosamente")

## Conectar LangChain a la base de datos

LangChain proporciona la clase `SQLDatabase` que actúa como un wrapper sobre nuestra base de datos. Esto le permite al agente inspeccionar las tablas, el esquema y ejecutar consultas.

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///tienda_videojuegos.db")

print("📋 Tablas disponibles:", db.get_usable_table_names())

In [ ]:
# Inspeccionar el esquema que el agente verá
print(db.get_table_info())

In [ ]:
# Probar una consulta directa para confirmar que funciona
resultado = db.run("SELECT titulo, precio FROM videojuegos WHERE precio > 50 ORDER BY precio DESC;")
print(resultado)

## Definir el contexto y el tool

Definimos un `RuntimeContext` para **inyectar** la conexión a la base de datos en el tool del agente. Esto permite que el tool acceda a la BD sin usar variables globales.

El tool `ejecutar_sql` recibe una consulta SQL y la ejecuta. El agente decidirá cuándo y cómo usarlo.

In [ ]:
from dataclasses import dataclass
from langchain_community.utilities import SQLDatabase


@dataclass
class RuntimeContext:
    """Contexto que se inyecta al agente en tiempo de ejecución."""
    db: SQLDatabase

In [ ]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def ejecutar_sql(consulta: str) -> str:
    """Ejecuta una consulta SQL de tipo SELECT en la base de datos SQLite y devuelve los resultados."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(consulta)
    except Exception as e:
        return f"Error: {e}"

## Crear el SQL Agent

Usamos `create_agent` con un system prompt que guía al agente a:
1. Descubrir las tablas y su esquema antes de escribir consultas
2. Usar solo consultas `SELECT` (solo lectura)
3. Responder siempre en español

El agente **no conoce el esquema de antemano** — debe descubrirlo por sí mismo usando el tool.

In [ ]:
PROMPT_SISTEMA = """Eres un analista experto en bases de datos SQLite.

Reglas:
- Piensa paso a paso.
- Cuando necesites datos, usa el tool `ejecutar_sql` con UNA consulta SELECT.
- Solo lectura: NO uses INSERT, UPDATE, DELETE, ALTER, DROP, CREATE, REPLACE ni TRUNCATE.
- Limita a 5 filas de salida a menos que el usuario pida explícitamente más.
- Si el tool devuelve 'Error:', revisa el SQL e inténtalo de nuevo.
- Prefiere listas explícitas de columnas; evita SELECT *.
- Responde siempre en español.
"""

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

agente = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[ejecutar_sql],
    system_prompt=PROMPT_SISTEMA,
    context_schema=RuntimeContext,
)

print("- Agente SQL creado")

In [ ]:
# Visualizar el grafo ReAct del agente
from IPython.display import Image, display

display(Image(agente.get_graph(xray=True).draw_mermaid_png()))

## Consultas al agente

Hagamos preguntas en lenguaje natural. Observa cómo el agente:
1. Descubre las tablas disponibles
2. Consulta el esquema
3. Genera y ejecuta la consulta SQL
4. Responde con base en los resultados

Usamos `agent.stream` para ver **cada paso** del ciclo ReAct (razonamiento → acción → observación). Puedes ver los traces completos en [LangSmith](https://smith.langchain.com).

In [ ]:
# Pregunta simple — el agente debe descubrir las tablas primero
pregunta = "¿Cuántos videojuegos hay en la tienda?"

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

In [ ]:
# Pregunta con filtro — necesita inspeccionar el esquema
pregunta = "¿Cuáles son los juegos de RPG y cuál tiene mejor calificación?"

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

In [ ]:
# Pregunta con JOIN y agregación
pregunta = "¿Cuál es el videojuego más vendido y cuántas unidades totales se vendieron?"

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

In [ ]:
# Pregunta analítica compleja
pregunta = "¿Cuánto dinero en ventas generó cada categoría? Ordénalas de mayor a menor ingreso."

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

## Agente con memoria

Podemos agregar un **checkpointer** para que el agente recuerde el contexto de la conversación. Esto permite hacer preguntas de seguimiento como *"¿y cuál fue el más barato?"* sin repetir contexto.

`MemorySaver` guarda el historial en RAM — ideal para demos y labs.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memoria = MemorySaver()

agente_con_memoria = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[ejecutar_sql],
    system_prompt=PROMPT_SISTEMA,
    context_schema=RuntimeContext,
    checkpointer=memoria,
)

config = {"configurable": {"thread_id": "tienda-01"}}

print("- Agente con memoria creado")

In [ ]:
# Primera pregunta
pregunta = "¿Qué juegos salieron en 2023?"

for paso in agente_con_memoria.stream(
    {"messages": pregunta},
    config=config,
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

In [ ]:
# Pregunta de seguimiento — usa el contexto previo
pregunta = "¿Y de esos cuál fue el más vendido?"

for paso in agente_con_memoria.stream(
    {"messages": pregunta},
    config=config,
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

## Actividad

### Ejercicio 1: Haz tus propias consultas
Escribe al menos **3 preguntas** en lenguaje natural sobre la tienda de videojuegos. Intenta que al menos una requiera un JOIN y otra una agregación (`SUM`, `COUNT`, `AVG`, etc.).

In [ ]:
# Pregunta 1 — JOIN: videojuegos con más ventas totales
pregunta = "¿Cuáles son los 3 videojuegos con más unidades vendidas en total y cuántas vendieron?"

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

In [ ]:
# Pregunta 2 — Agregación: precio promedio por categoría
pregunta = "¿Cuál es el precio promedio de los videojuegos por categoría? Ordena de mayor a menor precio."

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

In [ ]:
# Pregunta 3 — JOIN + filtro: juegos con alta calificación y sus ventas totales
pregunta = "¿Qué videojuegos tienen calificación mayor a 9.0 y cuántas unidades vendieron en total?"

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()

### Ejercicio 2: Agrega más datos

Agrega al menos **5 videojuegos nuevos** y **nuevas ventas** a la base de datos, luego hazle preguntas al agente sobre los datos nuevos.

In [ ]:
import sqlite3

conexion = sqlite3.connect("tienda_videojuegos.db")
cursor = conexion.cursor()

cursor.executescript("""
INSERT OR IGNORE INTO videojuegos VALUES (11, 'Red Dead Redemption 2', 1, 59.99, 2018, 9.7);
INSERT OR IGNORE INTO videojuegos VALUES (12, 'Cyberpunk 2077', 1, 39.99, 2020, 8.6);
INSERT OR IGNORE INTO videojuegos VALUES (13, 'Stardew Valley', 5, 14.99, 2016, 9.2);
INSERT OR IGNORE INTO videojuegos VALUES (14, 'Hades', 1, 24.99, 2020, 9.4);
INSERT OR IGNORE INTO videojuegos VALUES (15, 'It Takes Two', 3, 29.99, 2021, 9.3);

INSERT OR IGNORE INTO ventas VALUES (14, 11, 130, '2025-02-10');
INSERT OR IGNORE INTO ventas VALUES (15, 12, 270, '2025-01-18');
INSERT OR IGNORE INTO ventas VALUES (16, 13, 190, '2025-03-01');
INSERT OR IGNORE INTO ventas VALUES (17, 14, 145, '2025-02-25');
INSERT OR IGNORE INTO ventas VALUES (18, 15, 110, '2025-03-07');
""")

conexion.commit()
conexion.close()

# Recargar la conexión de LangChain
db = SQLDatabase.from_uri("sqlite:///tienda_videojuegos.db")
print("- 5 videojuegos y 5 ventas agregados. Conexión recargada.")

In [ ]:
# Pregunta sobre los datos nuevos
pregunta = "¿Cuáles son los videojuegos agregados recientemente (IDs 11 al 15) y cuál tiene la mejor calificación?"

for paso in agente.stream(
    {"messages": pregunta},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    paso["messages"][-1].pretty_print()